In [3]:
import sys
sys.path.append("../../")

import numpy as np

#
# TO BE USED WITH godot/mecanum_4w
#

from lib.system.mecanum import *
from lib.dds.dds import *
from lib.utils.time import *
from lib.system.controllers import *
from lib.system.trajectory import *
from lib.data.dataplot import *

class MecanumWheelsController:
    
    def __init__(self, _kp, _ki, _sat):
        self.controllers = []
        self.torques = [0,0,0,0]
        for i in range(4):
            self.controllers.append(PID_Controller(_kp, _ki, 0, _sat))
    
    def evaluate(self, delta_t, w_target, w_current):
        for i in range(4):
            self.torques[i] = self.controllers[i].evaluate(delta_t, w_target[i] - w_current[i])
        return self.torques
    
class MecanumPositionController:
    
    def __init__(self, _kp, _vmax):
        self.p_x = PID_Controller(_kp, 0, 0, _vmax)
        self.p_y = PID_Controller(_kp, 0, 0, _vmax)

    def evaluate(self, delta_t, target_pos, current_pos):
        v_x = self.p_x.evaluate(delta_t, target_pos[0] - current_pos[0])
        v_y = self.p_y.evaluate(delta_t, target_pos[1] - current_pos[1])
        return (v_x, v_y)
        
        
dds = DDS()
dds.start()

dds.subscribe(['tick'])

cart = MecanumFourWheels(10, # mass, 10kg 
                         0.7, # wheelbase
                         0.8, # lin-friction 
                         0.9, # ang-friction 
                         0.03) # 3cm radius motion wheel 
cart.set_pose(0,0,math.radians(-60))

wheel_controllers = MecanumWheelsController(0.05, 0.01, 5.0)
position_controllers = MecanumPositionController(1.0, 1.0)
trajectory = StraightLine2DMotion(1.0, 2.0, 2.0)

x_final = 0.8
y_final = -0.5
w_target = 0.0

pose = cart.get_pose()
trajectory.start_motion(pose[:2], [x_final, y_final])


t = Time()
t.start()
while t.get() < 10:

    #dds.wait('tick')
    t.sleep(0.001)
    delta_t = t.elapsed()

    pose = cart.get_pose()
    (x_target, y_target) = trajectory.evaluate(delta_t)
    (vx_target, vy_target) = position_controllers.evaluate(delta_t, [x_target, y_target], pose[:2])
    
    vx_local_target = vx_target * math.cos(pose[2]) + vy_target * math.sin(pose[2])
    vy_local_target = -vx_target * math.sin(pose[2]) + vy_target * math.cos(pose[2])
    v_local_targets = np.array( [vx_local_target, vy_local_target, w_target])
    
    w_targets = cart.ik_matrix @ v_local_targets
    w = cart.get_wheel_speed()
    torque = wheel_controllers.evaluate(delta_t, w_targets.flatten().tolist(), w)

    cart.evaluate(delta_t, torque[0], torque[1], torque[2], torque[3])

    dds.publish('X', pose[0], DDS.DDS_TYPE_FLOAT)
    dds.publish('Y', pose[1], DDS.DDS_TYPE_FLOAT)
    dds.publish('Theta', pose[2], DDS.DDS_TYPE_FLOAT)
    
    dds.publish('w1', w[0], DDS.DDS_TYPE_FLOAT)
    dds.publish('w2', w[1], DDS.DDS_TYPE_FLOAT)
    dds.publish('w3', w[2], DDS.DDS_TYPE_FLOAT)
    dds.publish('w4', w[3], DDS.DDS_TYPE_FLOAT)

dds.publish('w1', 0, DDS.DDS_TYPE_FLOAT)
dds.publish('w2', 0, DDS.DDS_TYPE_FLOAT)
dds.publish('w3', 0, DDS.DDS_TYPE_FLOAT)
dds.publish('w4', 0, DDS.DDS_TYPE_FLOAT)
dds.stop()

